# ARC NeuroGolf static ONNX solver 09- localish_recolor

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
required={'onnx':'onnx','onnxruntime':'onnxruntime','onnxsim':'onnxsim','torch':'torch','numpy':'numpy'}
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])


import json, os, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort
from onnxsim import simplify

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 79.2 MB/s eta 0:00:00


In [4]:
task_id = 'task208'
CH=10; H=W=30

In [5]:
class Task208StaticGraph(nn.Module):
    def __init__(self):
        super().__init__()
        inners=[]; rings=[]; areas=[]; ringareas=[]
        for h in range(2,6):
            for w in range(2,6):
                inner=torch.zeros(1,7,7); inner[:,1:h+1,1:w+1]=1.0
                ring=torch.zeros(1,7,7); ring[:,0:h+2,0:w+2]=1.0; ring[:,1:h+1,1:w+1]=0.0
                inners.append(inner); rings.append(ring); areas.append(h*w); ringareas.append(2*h+2*w+4)
        self.register_buffer('inner_k', torch.stack(inners,0))
        self.register_buffer('ring_k', torch.stack(rings,0))
        self.register_buffer('areas', torch.tensor(areas,dtype=torch.float32).view(1,16,1,1))
        self.register_buffer('ringareas', torch.tensor(ringareas,dtype=torch.float32).view(1,16,1,1))
    def forward(self,x):
        active=(x.sum(1,keepdim=True)>0).float()
        black=x[:,0:1]
        hole=(F.conv2d(black, self.inner_k)==self.areas).float()
        add=[torch.zeros_like(black) for _ in range(CH)]
        out=[x[:,k:k+1].clone() for k in range(CH)]
        for k in range(1,CH):
            ringcnt=F.conv2d(x[:,k:k+1], self.ring_k)
            complete=hole*(ringcnt==self.ringareas).float()
            has=(complete.sum((2,3),keepdim=True)>0).float()
            draw=hole*has
            add[k]=F.conv_transpose2d(draw, self.ring_k).clamp(0,1)
        occadd=sum(add[1:]).clamp(0,1)
        for k in range(1,CH):
            out[k]=(out[k]*(1-occadd)+add[k]).clamp(0,1)
        occ=sum(out[1:]).clamp(0,1)
        out[0]=(1-occ)*active
        return torch.cat(out,1)


In [6]:
def onehot(grid):
    arr=np.array(grid,dtype=np.int64); h,w=arr.shape
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    for k in range(CH): x[0,k,:h,:w]=(arr==k)
    return x,h,w

def inspect(path):
    m=onnx.load(str(path)); ops={}
    for n in m.graph.node: ops[n.op_type]=ops.get(n.op_type,0)+1
    return {'size':os.path.getsize(path),'forbidden':[op for op in ['Loop','Scan','NonZero','Unique','Script','Function'] if ops.get(op,0)],'risk':[op for op in ['Shape','Range','Expand','Gather','ScatterND','ConstantOfShape','Resize','Tile'] if ops.get(op,0)]}

def validate(path,task):
    sess=ort.InferenceSession(str(path),providers=['CPUExecutionProvider']); rep={}
    for sec in ['train','test','arc-gen']:
        ok=0
        for ex in task[sec]:
            x,h,w=onehot(ex['input']); pred=sess.run(None,{'input':x})[0].argmax(1)[0,:h,:w]
            ok+=int(np.array_equal(pred,np.array(ex['output'])))
        rep[sec]=(ok,len(task[sec]))
    return rep

In [7]:

class Task208StaticGraph(nn.Module):
    def __init__(self):
        super().__init__()
        inners=[]; rings=[]; areas=[]; ringareas=[]
        for h in range(2,6):
            for w in range(2,6):
                inner=torch.zeros(1,7,7); inner[:,1:h+1,1:w+1]=1.0
                ring=torch.zeros(1,7,7); ring[:,0:h+2,0:w+2]=1.0; ring[:,1:h+1,1:w+1]=0.0
                inners.append(inner); rings.append(ring); areas.append(h*w); ringareas.append(2*h+2*w+4)
        self.register_buffer('inner_k', torch.stack(inners,0))
        self.register_buffer('ring_k', torch.stack(rings,0))
        self.register_buffer('areas', torch.tensor(areas,dtype=torch.float32).view(1,16,1,1))
        self.register_buffer('ringareas', torch.tensor(ringareas,dtype=torch.float32).view(1,16,1,1))
    def forward(self,x):
        active=(x.sum(1,keepdim=True)>0).float()
        black=x[:,0:1]
        hole=(F.conv2d(black, self.inner_k)==self.areas).float()
        add=[torch.zeros_like(black) for _ in range(CH)]
        out=[x[:,k:k+1].clone() for k in range(CH)]
        for k in range(1,CH):
            ringcnt=F.conv2d(x[:,k:k+1], self.ring_k)
            complete=hole*(ringcnt==self.ringareas).float()
            has=(complete.sum((2,3),keepdim=True)>0).float()
            draw=hole*has
            add[k]=F.conv_transpose2d(draw, self.ring_k).clamp(0,1)
        occadd=sum(add[1:]).clamp(0,1)
        for k in range(1,CH):
            out[k]=(out[k]*(1-occadd)+add[k]).clamp(0,1)
        occ=sum(out[1:]).clamp(0,1)
        out[0]=(1-occ)*active
        return torch.cat(out,1)



In [8]:
model=Task208StaticGraph().eval(); 
path=Path(f'{task_id}.onnx')
torch.onnx.export(model, torch.zeros(1,CH,H,W), str(path), 
                  input_names=['input'], output_names=['output'], 
                  opset_version=13, dynamo=False)

m=onnx.load(str(path)); m.ir_version=min(m.ir_version,7); 
onnx.save(m,str(path)); onnx.checker.check_model(str(path))


task=json.load(open(Path(COMPETITION)/f'{task_id}.json'))
rep=validate(path,task); arch=inspect(path)
print(rep); print(arch)
assert rep['train'][0]==rep['train'][1] and rep['test'][0]==rep['test'][1] and rep['arc-gen'][0]==rep['arc-gen'][1]
assert arch['size']<1_400_000 and not arch['forbidden'] and not arch['risk']



/tmp/ipykernel_16/3529087907.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, torch.zeros(1,CH,H,W), str(path),


{'train': (3, 3), 'test': (1, 1), 'arc-gen': (262, 262)}
{'size': 28084, 'forbidden': [], 'risk': []}


In [9]:
with zipfile.ZipFile(Path.cwd()/'submission.zip','w',zipfile.ZIP_DEFLATED) as z: 
    z.write(path,f'{task_id}.onnx')
print('wrote submission.zip')

wrote submission.zip
